In [6]:
import time

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.tree import DecisionTreeRegressor

import optuna

from lineartree import (
    LinearTreeRegressor,
    SobolevForestRegressor,
)
import numpy as np
from sklearn.preprocessing import SplineTransformer
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.model_selection import KFold, cross_val_score

In [7]:
class TensorSplineRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, n_knots=6, degree=3, knots="quantile", r_alpha=None):
        self.n_knots = n_knots
        self.degree = degree
        self.knots = knots
        self.r_alpha = r_alpha

    def _basis(self, X, alpha):
        blocks = [
            spline(X[:, j], nu=alpha[j])
            for j, spline in enumerate(self.spline_.bsplines_)
        ]

        basis = blocks[0]
        for block in blocks[1:]:
            basis = np.einsum(
                "ni,nj->nij", basis, block
            ).reshape(len(X), -1)

        return basis

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.n_features_in_ = X.shape[1]

        self.spline_ = SplineTransformer(
            n_knots=self.n_knots,
            degree=self.degree,
            knots=self.knots,
            include_bias=True,
            extrapolation="continue",
        ).fit(X)

        basis = self._basis(X, (0,) * self.n_features_in_)

        if self.r_alpha is None:
            self.model_ = LinearRegression(
                fit_intercept=False
            ).fit(basis, y)
        else:
            self.model_ = Ridge(
                fit_intercept=False,
                alpha=self.r_alpha,
            ).fit(basis, y)

        return self

    def predict(self, X, alpha=None):
        check_is_fitted(self, "model_")
        X = check_array(X)

        alpha = (
            (0,) * self.n_features_in_
            if alpha is None
            else alpha
        )

        if len(alpha) != self.n_features_in_:
            raise ValueError("alpha must have one entry per feature")

        return self._basis(X, alpha) @ self.model_.coef_

In [8]:
def tune_forest(X, Y, k, T, tree_params, forest_params):
    space = {
        f"estimator__{p}": d for p, d in tree_params.items()
    } | forest_params

    model = SobolevForestRegressor(LinearTreeRegressor(
        Ridge(fit_intercept=False), derivative_weights=[0.0, 0.0]
    ))
    folds = KFold(k, shuffle=True, random_state=0)
    study = optuna.create_study()

    for _ in range(T):
        trial = study.ask(space)
        score = cross_val_score(
            model.set_params(**trial.params), X, Y,
            cv=folds, scoring="neg_mean_squared_error",
        ).mean()
        study.tell(trial, -score)

    return model.set_params(**study.best_params).fit(X, Y), study

def tune_spline(X, Y, k, T, spline_params):
    model = TensorSplineRegressor()
    folds = KFold(k, shuffle=True, random_state=0)
    study = optuna.create_study()

    for _ in range(T):
        trial = study.ask(spline_params)
        score = cross_val_score(
            model.set_params(**trial.params), X, Y,
            cv=folds, scoring="neg_mean_squared_error",
        ).mean()
        study.tell(trial, -score)

    return model.set_params(**study.best_params).fit(X, Y), study

In [20]:
dgp = "smooth_regimes"  # or "regime_mixed"

from scipy.special import expit

def gate(x, c, k=25):
    s = expit(k * (x - c))
    ds = k * s * (1 - s)
    return s, ds, k * ds * (1 - 2 * s)

# Columns: level, u² coefficient, v² coefficient, uv coefficient
regime_coef = np.array([
    [ 0,   8,  -4,  6],
    [ 7, -10,   6, -8],
    [-6,   5,  12,  4],
    [10,  -6, -10,  8],
    [-8,  12,   4, -6],
    [ 4,  -8,  10, -4],
])

def f(x):
    u, v = x.T

    if dgp == "sine":
        su, sv = np.sin(2 * np.pi * u), np.sin(2 * np.pi * v)
        return su + sv + 0.5 * su * sv

    if dgp == "quartic":
        return (u**2 + v - 1)**2 + (u + v**2 - 1)**2

    if dgp == "exponential":
        return 2 * np.exp(2 * u * v - 2)

    if dgp == "regime_quadratic":
        r = np.searchsorted([0.35, 0.70], u) + 3 * (v > 0.55)
        m, a, b, c = regime_coef[r].T
        return (
            m + a * (u - 0.5)**2 + b * (v - 0.5)**2
            + c * (u - 0.5) * (v - 0.5)
        )

    if dgp == "regime_mixed":
        return np.select(
            [u < 0.4, v < 0.6],
            [
                6 + 3 * np.sin(2 * np.pi * u),
                -6 + 8 * (v - 0.3)**2 + 4 * u * v,
            ],
            default=(
                2 + 3 * np.exp(2 * (u - 0.7))
                + 20 * (v - 0.8)**3
            ),
        )

    if dgp == "smooth_regimes":
        s = gate(u, 0.4)[0]
        t = gate(v, 0.6)[0]
        return (
            u**2 + v**2
            + s * (3 - 3 * v**2)
            + t * (-5 + 4 * u**2)
        )

    if dgp == "gated_sine":
        s = gate(u, 0.6)[0]
        return u**2 + v + s * (3 + 2 * np.sin(4 * np.pi * v))

    raise ValueError(dgp)

def gradient_f(x):
    u, v = x.T

    if dgp == "smooth_regimes":
        s, ds, _ = gate(u, 0.4)
        t, dt, _ = gate(v, 0.6)

        du = 2 * u + ds * (3 - 3 * v**2) + 8 * u * t
        dv = 2 * v - 6 * v * s + dt * (-5 + 4 * u**2)

    elif dgp == "gated_sine":
        s, ds, _ = gate(u, 0.6)
        w = 4 * np.pi

        du = 2 * u + ds * (3 + 2 * np.sin(w * v))
        dv = 1 + 2 * w * s * np.cos(w * v)

    elif dgp == "exponential":
        g = f(x)
        du, dv = 2 * v * g, 2 * u * g

    else:
        raise ValueError(dgp)

    return np.column_stack((du, dv))


def hessian_f(x):
    u, v = x.T
    H = np.empty((len(x), 2, 2))

    if dgp == "sine":
        w = 2 * np.pi
        su, sv = np.sin(w * u), np.sin(w * v)
        H[:, 0, 0] = -w**2 * su * (1 + 0.5 * sv)
        H[:, 1, 1] = -w**2 * sv * (1 + 0.5 * su)
        H[:, 0, 1] = H[:, 1, 0] = (
            0.5 * w**2 * np.cos(w * u) * np.cos(w * v)
        )

    elif dgp == "quartic":
        H[:, 0, 0] = 12 * u**2 + 4 * v - 2
        H[:, 1, 1] = 12 * v**2 + 4 * u - 2
        H[:, 0, 1] = H[:, 1, 0] = 4 * (u + v)

    elif dgp == "exponential":
        g = f(x)
        H[:, 0, 0] = 4 * v**2 * g
        H[:, 1, 1] = 4 * u**2 * g
        H[:, 0, 1] = H[:, 1, 0] = (2 + 4 * u * v) * g

    elif dgp == "regime_quadratic":
        r = np.searchsorted([0.35, 0.70], u) + 3 * (v > 0.55)
        _, a, b, c = regime_coef[r].T

        H[:, 0, 0] = 2 * a
        H[:, 1, 1] = 2 * b
        H[:, 0, 1] = H[:, 1, 0] = c

    elif dgp == "regime_mixed":
        regions = [u < 0.4, v < 0.6]

        H[:, 0, 0] = np.select(
            regions,
            [-12 * np.pi**2 * np.sin(2 * np.pi * u), 0],
            default=12 * np.exp(2 * (u - 0.7)),
        )
        H[:, 1, 1] = np.select(
            regions, [0, 16], default=120 * (v - 0.8)
        )
        H[:, 0, 1] = H[:, 1, 0] = np.select(
            regions, [0, 4], default=0
        )

    elif dgp == "smooth_regimes":
        s, ds, dds = gate(u, 0.4)
        t, dt, ddt = gate(v, 0.6)

        H[:, 0, 0] = 2 + dds * (3 - 3 * v**2) + 8 * t
        H[:, 1, 1] = 2 - 6 * s + ddt * (-5 + 4 * u**2)
        H[:, 0, 1] = H[:, 1, 0] = -6 * v * ds + 8 * u * dt

    elif dgp == "gated_sine":
        s, ds, dds = gate(u, 0.6)
        w = 4 * np.pi

        H[:, 0, 0] = 2 + dds * (3 + 2 * np.sin(w * v))
        H[:, 1, 1] = -2 * w**2 * s * np.sin(w * v)
        H[:, 0, 1] = H[:, 1, 0] = 2 * w * ds * np.cos(w * v)

    else:
        raise ValueError(dgp)

    return H


x_test_fixed = np.random.default_rng(2026).uniform(size=(50_000, 2))

def make_mc(seeds=range(100), n_train=1_000, sigma=0.2):
    mc = {}
    y_test = f(x_test_fixed)

    for seed in seeds:
        rng = np.random.default_rng(seed)
        x_train = rng.uniform(size=(n_train, 2))
        y_train = f(x_train) + rng.normal(scale=sigma, size=n_train)
        mc[seed] = (x_train, y_train, x_test_fixed, y_test)

    return mc

In [21]:
mse_20, mse_11, mse_02 = [[], []], [[], []], [[], []]
mae_20, mae_11, mae_02 = [[], []], [[], []], [[], []]
mse_10, mse_01 = [[], []], [[], []]
mae_10, mae_01 = [[], []], [[], []]
mse_00, mae_00 = [[], []], [[], []]

D = optuna.distributions

tree_params = {
    "max_depth": D.IntDistribution(2, 4),
    "min_samples_leaf": D.IntDistribution(20, 60),
    "max_features": D.CategoricalDistribution([2]),
    "base_estimator__alpha": D.FloatDistribution(1e-5, 1e-2, log=True),
}

forest_params = {
    "max_samples": D.FloatDistribution(0.6, 0.9),
    "n_estimators": D.CategoricalDistribution([300]),
    "n_jobs": D.CategoricalDistribution([28]),
    "random_state": D.CategoricalDistribution([0]),
}

spline_params = {
    "n_knots": D.IntDistribution(3, 5),
    "knots": D.CategoricalDistribution(["uniform", "quantile"]),
    "r_alpha": D.FloatDistribution(1e-5, 0.1, log=True),
}

i = 0
for seed, data in make_mc(seeds=range(20), n_train=3000).items():
    simulation_time = time.perf_counter()
    print(i + 1)
    x_train, y_train, x_test, y_test = data

    forest, forest_study = tune_forest(
        x_train, y_train, 3, 30, tree_params, forest_params
    )
    spline, spline_study = tune_spline(
        x_train, y_train, 3, 30, spline_params
    )

    gradient_test = gradient_f(x_test)
    hessian_test = hessian_f(x_test)

    metrics = (
        ("f",   (0, 0), y_test,                 mse_00, mae_00),
        ("d10", (1, 0), gradient_test[:, 0],    mse_10, mae_10),
        ("d01", (0, 1), gradient_test[:, 1],    mse_01, mae_01),
        ("d20", (2, 0), hessian_test[:, 0, 0], mse_20, mae_20),
        ("d11", (1, 1), hessian_test[:, 0, 1], mse_11, mae_11),
        ("d02", (0, 2), hessian_test[:, 1, 1], mse_02, mae_02),
    )

    for j, (name, model) in enumerate(
        (("forest", forest), ("spline", spline))
    ):
        for label, alpha, truth, mse, mae in metrics:
            error = model.predict(x_test, alpha=alpha) - truth

            mse[j].append(np.mean(error**2))
            mae[j].append(np.mean(np.abs(error)))

            print(
                f"MSE {name} {label}: {np.mean(mse[j])} "
                f"MAE {name} {label}: {np.mean(mae[j])}"
            )
    print('Forest best params:', forest_study.best_params)
    print('Time elaspsed:', time.perf_counter() - simulation_time)
    print()

    i += 1

[I 2026-11-04 16:28:39,007] A new study created in memory with name: no-name-1e37852b-0f79-446c-9fe2-166663c3be8a


1


[I 2026-11-04 16:40:38,471] A new study created in memory with name: no-name-a9242882-33a3-41a9-993b-95000e6e4bf2


MSE forest f: 0.0020070547287132003 MAE forest f: 0.033795502913357374
MSE forest d10: 0.6919368477153673 MAE forest d10: 0.5755271755075005
MSE forest d01: 1.6414562645700022 MAE forest d01: 0.7651637533112106
MSE forest d20: 1962.4734192246615 MAE forest d20: 22.922646822940873
MSE forest d11: 22.057424763432714 MAE forest d11: 3.2785472796848185


[I 2026-11-04 16:41:00,930] A new study created in memory with name: no-name-1138ae8c-1ddd-4d15-b0d8-9d9fb7a680a6


MSE forest d02: 5530.348469578374 MAE forest d02: 38.184412432968436
MSE spline f: 0.018379097361316918 MAE spline f: 0.10578634326221609
MSE spline d10: 3.2596724446148646 MAE spline d10: 1.3132732652763706
MSE spline d01: 9.840561929641543 MAE spline d01: 2.2840783371378204
MSE spline d20: 2072.8064131189353 MAE spline d20: 31.908923755674177
MSE spline d11: 41.02909818591265 MAE spline d11: 4.527718810132359
MSE spline d02: 6244.949814706148 MAE spline d02: 56.60955674229095
Forest best params: {'estimator__max_depth': 3, 'estimator__min_samples_leaf': 35, 'estimator__max_features': 2, 'estimator__base_estimator__alpha': 8.176095551566337e-05, 'max_samples': 0.6221655029486013, 'n_estimators': 300, 'n_jobs': 28, 'random_state': 0}
Time elaspsed: 741.922413799999

2


... (21 outputs left out)

MSE forest f: 0.0017730822442799255 MAE forest f: 0.03205741985086611
MSE forest d10: 0.8619146493176787 MAE forest d10: 0.5965064082824284
MSE forest d01: 1.2741790798522363 MAE forest d01: 0.6819788982914827
MSE forest d20: 2165.832598074287 MAE forest d20: 23.372304499915213
MSE forest d11: 27.24060336253883 MAE forest d11: 3.341622665334062


[I 2026-11-04 18:00:19,127] A new study created in memory with name: no-name-05f47840-a722-4ba2-9fe5-1d44873b2e2d


MSE forest d02: 5198.438908801841 MAE forest d02: 36.678300861574485
MSE spline f: 0.01838552865783394 MAE spline f: 0.10605264223933156
MSE spline d10: 3.2823005443010382 MAE spline d10: 1.29550070496867
MSE spline d01: 10.215135560430225 MAE spline d01: 2.316667781784374
MSE spline d20: 2061.215681008866 MAE spline d20: 31.490543842261935
MSE spline d11: 39.2852253332607 MAE spline d11: 4.462208097626589
MSE spline d02: 6473.539533796023 MAE spline d02: 57.54775809596275
Forest best params: {'estimator__max_depth': 4, 'estimator__min_samples_leaf': 30, 'estimator__max_features': 2, 'estimator__base_estimator__alpha': 0.0004896693032970047, 'max_samples': 0.6262226751099031, 'n_estimators': 300, 'n_jobs': 28, 'random_state': 0}
Time elaspsed: 897.4440972000011

8


KeyboardInterrupt: 

In [19]:
import pickle

results = {
    f"{metric}_{alpha}": globals()[f"{metric}_{alpha}"]
    for metric in ("mse", "mae")
    for alpha in ("00", "10", "01", "20", "11", "02")
}

with open("mse_mae_exponential.pkl", "wb") as file:
    pickle.dump(results, file, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open("mse_mae_smooth_regimes.pkl", "rb") as file:
    results = pickle.load(file)

results["mse_20"][0]  # Forest d20 MSE across repetitions
results["mse_20"][1]  # Spline d20 MSE across repetitions